In [6]:
# ============================================================
# ViT-B/16 Baseline Training for BCN20000 (Image-only)
# Complete Code with Auto-Detect and Fixed ROC-AUC
# ============================================================

import os
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score
from sklearn.preprocessing import label_binarize
import timm
from tqdm.notebook import tqdm
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# -------------------------------
# 1. Mount Google Drive
# -------------------------------
from google.colab import drive
drive.mount('/content/drive')

# -------------------------------
# 2. Auto-Detect Dataset
# -------------------------------
def find_dataset():
    """Auto-detect zip file or extracted dataset"""

    # Option A: Check if already extracted
    possible_dirs = [
        '/content/bcn20000/bcn_20k_train',
        '/content/bcn20000',
        '/content/drive/My Drive/SKIN paper/bcn20000',
    ]

    for dir_path in possible_dirs:
        if os.path.exists(dir_path):
            files = os.listdir(dir_path)
            if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files):
                print(f"✅ Found extracted images in: {dir_path}")
                return {'type': 'extracted', 'path': dir_path}

    # Option B: Find zip file
    zip_candidates = [
        '/content/drive/My Drive/SKIN paper/BCN_20k_train.zip',
        '/content/drive/My Drive/SKIN paper/bcn_20k_train.zip',
        '/content/drive/My Drive/BCN_20k_train.zip',
    ]

    for zip_path in zip_candidates:
        if os.path.exists(zip_path):
            print(f"✅ Found zip file: {zip_path}")
            return {'type': 'zip', 'path': zip_path}

    return None

dataset = find_dataset()

if dataset is None:
    print("❌ Dataset not found. Please upload it to your Drive.")
    raise FileNotFoundError("Dataset not found")

# -------------------------------
# 3. Set Paths
# -------------------------------
if dataset['type'] == 'zip':
    ZIP_PATH = dataset['path']
    EXTRACT_DIR = '/content/bcn20000'
    IMAGE_DIR = os.path.join(EXTRACT_DIR, 'bcn_20k_train')
    METADATA_FILE = '/content/drive/My Drive/SKIN paper/bcn_20k_train.csv'

    # Extract if needed
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    if not os.path.exists(IMAGE_DIR) or len(os.listdir(IMAGE_DIR)) < 100:
        print(f"Extracting {ZIP_PATH}... This may take 5-10 minutes.")
        with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
        print("✅ Extraction complete")
    else:
        print("✅ Dataset already extracted.")
else:
    # Already extracted
    IMAGE_DIR = dataset['path']
    EXTRACT_DIR = os.path.dirname(IMAGE_DIR)
    METADATA_FILE = '/content/drive/My Drive/SKIN paper/bcn_20k_train.csv'

# -------------------------------
# 4. Verify CSV
# -------------------------------
if not os.path.exists(METADATA_FILE):
    print("Searching for CSV...")
    for root, dirs, files in os.walk('/content/drive/My Drive'):
        if 'bcn_20k_train.csv' in files:
            METADATA_FILE = os.path.join(root, 'bcn_20k_train.csv')
            print(f"✅ Found CSV: {METADATA_FILE}")
            break

print(f"✅ Image dir: {IMAGE_DIR}")
print(f"✅ Metadata: {METADATA_FILE}")

# -------------------------------
# 5. Load Metadata
# -------------------------------
df = pd.read_csv(METADATA_FILE)
print(f"Loaded {len(df)} rows")
print("Columns:", df.columns.tolist())

# -------------------------------
# 6. Detect Columns
# -------------------------------
diagnosis_col = next((c for c in df.columns if c.lower() in ['diagnostic', 'diagnosis', 'class', 'lesion_type']), df.columns[0])
image_col = next((c for c in df.columns if c.lower() in ['image', 'image_id', 'filename', 'file_name', 'bcn_filename']), df.columns[1])
lesion_col = next((c for c in df.columns if c.lower() in ['lesion_id', 'patient_id']), None)

print(f"Diagnosis: {diagnosis_col}, Image: {image_col}, Lesion: {lesion_col if lesion_col else 'None'}")

# -------------------------------
# 7. Map Labels
# -------------------------------
class_mapping = {
    'ak': 0, 'actinic keratosis': 0,
    'bcc': 1, 'basal cell carcinoma': 1,
    'mel': 2, 'melanoma': 2,
    'met': 3, 'melanoma metastasis': 3,
    'nv': 4, 'nevus': 4,
    'sk': 5, 'seborrheic keratosis': 5
}

df['label'] = df[diagnosis_col].astype(str).str.lower().str.strip().map(class_mapping)

unknown = df[df['label'].isna()]
if len(unknown) > 0:
    print(f"⚠️ Dropping {len(unknown)} unknown labels: {unknown[diagnosis_col].unique()}")
    df = df.dropna(subset=['label']).reset_index(drop=True)

df['label'] = df['label'].astype(int)
print("Class distribution:\n", df['label'].value_counts().sort_index())

# -------------------------------
# 8. Build Image Paths
# -------------------------------
df['image_path'] = df[image_col].apply(lambda x: os.path.join(IMAGE_DIR, x))

def fix_path(p):
    if os.path.exists(p): return p
    base, ext = os.path.splitext(p)
    for e in ['.jpg', '.jpeg', '.png']:
        if os.path.exists(base + e): return base + e
    return p

df['image_path'] = df['image_path'].apply(fix_path)

missing = df[~df['image_path'].apply(os.path.exists)]
if len(missing) > 0:
    print(f"⚠️ {len(missing)} images missing. Dropping.")
    df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)

print(f"Final samples: {len(df)}")

# -------------------------------
# 9. Split Data
# -------------------------------
def split_data(df, lesion_col=None, test_size=0.1, val_size=0.1, random_state=42):
    if lesion_col and lesion_col in df.columns:
        lesions = df.groupby(lesion_col).agg({'label': 'first'}).reset_index()
        train_val_lesions, test_lesions = train_test_split(
            lesions, test_size=test_size, stratify=lesions['label'], random_state=random_state
        )
        train_lesions, val_lesions = train_test_split(
            train_val_lesions, test_size=val_size/(1-test_size),
            stratify=train_val_lesions['label'], random_state=random_state
        )
        train_df = df[df[lesion_col].isin(train_lesions[lesion_col])].reset_index(drop=True)
        val_df = df[df[lesion_col].isin(val_lesions[lesion_col])].reset_index(drop=True)
        test_df = df[df[lesion_col].isin(test_lesions[lesion_col])].reset_index(drop=True)
    else:
        train_df, temp_df = train_test_split(df, test_size=test_size+val_size,
                                             stratify=df['label'], random_state=random_state)
        val_df, test_df = train_test_split(temp_df, test_size=test_size/(test_size+val_size),
                                           stratify=temp_df['label'], random_state=random_state)
        train_df = train_df.reset_index(drop=True)
        val_df = val_df.reset_index(drop=True)
        test_df = test_df.reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = split_data(df, lesion_col)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print("Train class distribution:\n", train_df['label'].value_counts().sort_index())

# -------------------------------
# 10. Dataset Class
# -------------------------------
class BCN20000Dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['image_path']).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.df.iloc[idx]['label']

# -------------------------------
# 11. Transforms
# -------------------------------
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# -------------------------------
# 12. DataLoaders
# -------------------------------
BATCH_SIZE = 32  # Reduced to avoid OOM

train_loader = DataLoader(BCN20000Dataset(train_df, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(BCN20000Dataset(val_df, val_transform),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(BCN20000Dataset(test_df, val_transform),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"✅ DataLoaders ready. Batch size: {BATCH_SIZE}")

# -------------------------------
# 13. Model
# -------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=6).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)

print(f"✅ Model: ViT-B/16 with {sum(p.numel() for p in model.parameters()):,} params")

# -------------------------------
# 14. Training
# -------------------------------
best_loss = float('inf')
best_state = None
patience = 5
wait = 0

print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

for epoch in range(30):
    # Train
    model.train()
    train_loss, correct, total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}')
    for imgs, lbls in pbar:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
        _, pred = torch.max(out, 1)
        correct += (pred == lbls).sum().item()
        total += lbls.size(0)
        pbar.set_postfix({'Loss': f'{loss.item():.4f}'})

    train_acc = correct / total
    train_loss = train_loss / total

    # Validate
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = model(imgs)
            loss = criterion(out, lbls)
            val_loss += loss.item() * imgs.size(0)
            _, pred = torch.max(out, 1)
            val_correct += (pred == lbls).sum().item()
            val_total += lbls.size(0)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())

    val_acc = val_correct / val_total
    val_loss = val_loss / val_total
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    val_mcc = matthews_corrcoef(all_labels, all_preds)

    print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Loss={train_loss:.4f} | Val Acc={val_acc:.4f}, Loss={val_loss:.4f}, F1={val_f1:.4f}, MCC={val_mcc:.4f}")

    scheduler.step()

    if val_loss < best_loss:
        best_loss = val_loss
        best_state = model.state_dict().copy()
        wait = 0
        print("  ✅ Best model updated")
    else:
        wait += 1
        if wait >= patience:
            print(f"  ⏹️ Early stopping after {epoch+1} epochs")
            break

if best_state:
    model.load_state_dict(best_state)

print("\n✅ Training complete")

# -------------------------------
# 15. Test Evaluation (Fixed ROC-AUC)
# -------------------------------
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for imgs, lbls in tqdm(test_loader, desc='Testing'):
        imgs, lbls = imgs.to(device), lbls.to(device)
        out = model(imgs)
        probs = torch.softmax(out, dim=1)
        _, pred = torch.max(out, 1)
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average='macro')
test_mcc = matthews_corrcoef(all_labels, all_preds)

# ROC-AUC - handle missing classes
unique_labels = np.unique(all_labels)
print(f"\nClasses present in test set: {unique_labels}")

# Binarize only for classes that are present
all_labels_onehot = label_binarize(all_labels, classes=unique_labels)
test_probs_subset = np.array(all_probs)[:, unique_labels]

try:
    test_roc = roc_auc_score(all_labels_onehot, test_probs_subset,
                              average='macro', multi_class='ovr')
except ValueError as e:
    print(f"Warning: ROC-AUC could not be computed: {e}")
    test_roc = np.nan

f1_per_class = f1_score(all_labels, all_preds, average=None)

print("\n" + "="*60)
print("TEST RESULTS")
print("="*60)
print(f"Accuracy  : {test_acc:.4f}")
print(f"Macro-F1  : {test_f1:.4f}")
print(f"MCC       : {test_mcc:.4f}")
print(f"ROC-AUC   : {test_roc:.4f}")
print(f"Per-class F1: {f1_per_class}")
print("="*60)

# -------------------------------
# 16. Save Results
# -------------------------------
results_df = pd.DataFrame({
    'model': ['ViT-B/16'],
    'accuracy': [test_acc],
    'macro_f1': [test_f1],
    'mcc': [test_mcc],
    'roc_auc': [test_roc]
})
results_df.to_csv('/content/vit_b16_results.csv', index=False)
torch.save(best_state, '/content/vit_b16_best_model.pth')

# Copy to Drive
try:
    !cp /content/vit_b16_results.csv "/content/drive/My Drive/SKIN paper/"
    !cp /content/vit_b16_best_model.pth "/content/drive/My Drive/SKIN paper/"
    print("\n✅ Results saved to: /content/drive/My Drive/SKIN paper/")
except Exception as e:
    print(f"\n⚠️ Could not copy to Drive: {e}")
    print("Files saved locally at /content/")

print("\n📊 Results file: vit_b16_results.csv")
print("🧠 Model file: vit_b16_best_model.pth")
print("\n✅ ViT-B/16 baseline training complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Found extracted images in: /content/bcn20000/bcn_20k_train
✅ Image dir: /content/bcn20000/bcn_20k_train
✅ Metadata: /content/drive/My Drive/SKIN paper/bcn_20k_train.csv
Loaded 12413 rows
Columns: ['bcn_filename', 'age_approx', 'anatom_site_general', 'diagnosis', 'lesion_id', 'capture_date', 'sex', 'split']
Diagnosis: diagnosis, Image: bcn_filename, Lesion: lesion_id
⚠️ Dropping 1804 unknown labels: ['SCC' 'BKL' 'DF' 'VASC']
Class distribution:
 label
0     737
1    2809
2    2857
4    4206
Name: count, dtype: int64
Final samples: 10609
Train: 8484, Val: 1085, Test: 1040
Train class distribution:
 label
0     585
1    2231
2    2319
4    3349
Name: count, dtype: int64
✅ DataLoaders ready. Batch size: 32
Using: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

✅ Model: ViT-B/16 with 85,803,270 params

STARTING TRAINING


Epoch 1:   0%|          | 0/266 [00:00<?, ?it/s]

...

Epoch 20: Train Acc=0.7900, Loss=0.5317 | Val Acc=0.6829, Loss=0.8398, F1=0.6148, MCC=0.5471


Epoch 21:   0%|          | 0/266 [00:00<?, ?it/s]

Epoch 21: Train Acc=0.7962, Loss=0.5102 | Val Acc=0.6876, Loss=0.8153, F1=0.5905, MCC=0.5541
  ⏹️ Early stopping after 21 epochs

✅ Training complete


Testing:   0%|          | 0/33 [00:00<?, ?it/s]


Classes present in test set: [0 1 2 4]

TEST RESULTS
Accuracy  : 0.6663
Macro-F1  : 0.5344
MCC       : 0.5083
ROC-AUC   : 0.8583
Per-class F1: [0.14       0.72670807 0.5021645  0.76887872]

✅ Results saved to: /content/drive/My Drive/SKIN paper/

📊 Results file: vit_b16_results.csv
🧠 Model file: vit_b16_best_model.pth

✅ ViT-B/16 baseline training complete!
